<a href="https://colab.research.google.com/github/Silva015/tcc-2026/blob/main/ConvKANWithResNet_MLSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalação das implementações eficientes da rede KAN e bibliotecas de augmentation
!pip install git+https://github.com/Blealtan/efficient-kan -q
!pip install albumentations -q
!pip install efficient-kan

# Clonagem dos repositórios contendo os dados e códigos base
!git clone https://gitlab.com/lisa-unb/leguwoi.git -q
!git clone https://github.com/pedrogarciafreitas/FDCPUnBGunshotDB.git -q

import sys

import os
from glob import glob
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import copy
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, classification_report
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executando no dispositivo: {device}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
fatal: destination path 'leguwoi' already exists and is not an empty directory.
fatal: destination path 'FDCPUnBGunshotDB' already exists and is not an empty directory.
Executando no dispositivo: cuda


In [2]:
class GunshotDistanceDataset(Dataset):
    """
    Carrega apenas imagens de ENTRADA e atribui labels de distância (0: Encostado, 1: Curta, 2: Distância).
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        print("🔄 Mapeando imagens de distâncias na base de dados...")

        # 1. ENCOSTADO (Label 0)
        caminho_encostado = os.path.join(root_dir, 'database', 'ENTRADAS_EQX', 'ENCOSTADO', '*.JPG')
        for path in glob(caminho_encostado):
            self.image_paths.append(path)
            self.labels.append(0)

        # 2. CURTA DISTÂNCIA (Label 1)
        caminho_curta = os.path.join(root_dir, 'database', 'ENTRADAS_EQX', 'CURTA', '*.JPG')
        for path in glob(caminho_curta):
            self.image_paths.append(path)
            self.labels.append(1)

        # 3. À DISTÂNCIA (Label 2)
        caminho_distancia = os.path.join(root_dir, 'database', 'ENTRADAS_EQX', 'DISTANCIA', '*.JPG')
        for path in glob(caminho_distancia):
            self.image_paths.append(path)
            self.labels.append(2)

        print(f"✅ Concluído! Total de imagens de ENTRADA encontradas: {len(self.image_paths)}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image_np = np.array(image)
            augmented = self.transform(image=image_np)
            image = augmented['image']

        return image, label

In [3]:
import random

# Tamanho da imagem padronizado para 128x128 como nos experimentos originais de ConvKAN
IMG_SIZE = 128

transform_treino = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

transform_teste = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# 1. Configuração de Sementes
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

dataset_base_treino = GunshotDistanceDataset(root_dir='./FDCPUnBGunshotDB', transform=transform_treino)
dataset_base_teste  = GunshotDistanceDataset(root_dir='./FDCPUnBGunshotDB', transform=transform_teste)

# 2. Separação Semântica Estrita (Por Caso ID)
case_to_indices = {}
for idx, path in enumerate(dataset_base_treino.image_paths):
    filename = os.path.basename(path)
    case_id = filename[:9]
    if case_id not in case_to_indices:
        case_to_indices[case_id] = []
    case_to_indices[case_id].append(idx)

unique_cases = list(case_to_indices.keys())
cases_treino, cases_teste = train_test_split(unique_cases, test_size=0.2, random_state=RANDOM_SEED)

indices_treino = []
for case in cases_treino:
    indices_treino.extend(case_to_indices[case])

indices_teste = []
for case in cases_teste:
    indices_teste.extend(case_to_indices[case])

dataset_treino = Subset(dataset_base_treino, indices_treino)
dataset_teste  = Subset(dataset_base_teste, indices_teste)

trainloader = DataLoader(dataset_treino, batch_size=8, shuffle=True)
testloader  = DataLoader(dataset_teste, batch_size=8, shuffle=False)

print("\n" + "="*50)
print("🚀 SANITY CHECK DE VAZAMENTO DE DADOS (CÓDIGO)")
print("="*50)

train_cases_check = set([os.path.basename(dataset_base_treino.image_paths[i])[:9] for i in indices_treino])
test_cases_check  = set([os.path.basename(dataset_base_teste.image_paths[i])[:9] for i in indices_teste])
intersecao = train_cases_check.intersection(test_cases_check)

print(f"📊 Total de Casos Únicos de ENTRADA: {len(unique_cases)}")
print(f"   -> Casos no Treino: {len(train_cases_check)} casos ({len(dataset_treino)} imagens)")
print(f"   -> Casos no Teste:  {len(test_cases_check)} casos ({len(dataset_teste)} imagens)")

if len(intersecao) == 0:
    print("\n✅ SUCESSO! Zero interseção semântica identificada pelo código.")
else:
    print(f"\n❌ ALERTA DE VAZAMENTO! {len(intersecao)} casos vazados: {intersecao}")
print("="*50)

🔄 Mapeando imagens de distâncias na base de dados...
✅ Concluído! Total de imagens de ENTRADA encontradas: 1883
🔄 Mapeando imagens de distâncias na base de dados...
✅ Concluído! Total de imagens de ENTRADA encontradas: 1883

🚀 SANITY CHECK DE VAZAMENTO DE DADOS (CÓDIGO)
📊 Total de Casos Únicos de ENTRADA: 441
   -> Casos no Treino: 352 casos (1532 imagens)
   -> Casos no Teste:  89 casos (351 imagens)

✅ SUCESSO! Zero interseção semântica identificada pelo código.


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_24575/3692904553.py:12: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
/tmp/ipykernel_24575/3692904553.py:13: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),


In [4]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet152_Weights
from efficient_kan import KAN

class ConvKANWithResNet_MLSD(nn.Module):
    def __init__(self, freeze_backbone=True):
        super(ConvKANWithResNet_MLSD, self).__init__()

        # 1. Carrega a ResNet-152 pré-treinada na ImageNet
        self.backbone = models.resnet152(weights=ResNet152_Weights.IMAGENET1K_V1)

        # 2. Congela as convoluções para focar na camada KAN
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # 3. Substitui a camada linear pela KAN
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity() # Deixa o fluxo passar reto

        # Estrutura KAN: [Entrada (2048), Oculta (64), Saída(3 para Distância)]
        self.kan_head = KAN([in_features, 64, 3])

    def forward(self, x):
        features = self.backbone(x)
        return self.kan_head(features)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_mlsd = ConvKANWithResNet_MLSD(freeze_backbone=True).to(device)

print(f"🚀 ConvKANWithResNet (Híbrida) MLSD inicializada em {device.upper()}")
print(f"Parâmetros treináveis (apenas KAN): {sum(p.numel() for p in model_mlsd.parameters() if p.requires_grad):,}")

Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth


100%|██████████| 230M/230M [00:00<00:00, 244MB/s]


🚀 ConvKANWithResNet (Híbrida) MLSD inicializada em CUDA
Parâmetros treináveis (apenas KAN): 1,312,640


In [5]:
import copy
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

epochs = 60

# Pesos balanceados para as 3 distâncias
train_labels = [dataset_base_treino.labels[i] for i in indices_treino]
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
# Learning rate maior na KAN (1e-3) pois as convoluções estão congeladas
optimizer = torch.optim.AdamW(model_mlsd.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_acc = 0.0
best_model_wts = copy.deepcopy(model_mlsd.state_dict())

print(f"🔥 Iniciando Fine-tuning da ConvKANWithResNet (MLSD)...")

for epoch in range(epochs):
    model_mlsd.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_mlsd(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        running_loss += loss.item()

    acc_train = 100 * correct_train / total_train

    model_mlsd.eval()
    correct_test, total_test = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_mlsd(images)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    acc_test = 100 * correct_test / total_test
    scheduler.step()

    if acc_test > best_acc:
        best_acc = acc_test
        best_model_wts = copy.deepcopy(model_mlsd.state_dict())
        torch.save(best_model_wts, 'melhor_modelo_ConvKANWithResNet_MLSD.pth')
        ind = "🏆 NOVO RECORDE"
    else:
        ind = ""

    print(f"Época {epoch+1:02d} | Treino: {acc_train:.2f}% | Teste: {acc_test:.2f}% {ind}")

🔥 Iniciando Fine-tuning da ConvKANWithResNet (MLSD)...
Época 01 | Treino: 83.42% | Teste: 82.34% 🏆 NOVO RECORDE
Época 02 | Treino: 82.57% | Teste: 81.77% 
Época 03 | Treino: 75.65% | Teste: 77.21% 
Época 04 | Treino: 77.02% | Teste: 80.63% 
Época 05 | Treino: 80.74% | Teste: 67.81% 
Época 06 | Treino: 75.72% | Teste: 80.91% 
Época 07 | Treino: 72.78% | Teste: 79.49% 
Época 08 | Treino: 77.09% | Teste: 72.08% 
Época 09 | Treino: 76.04% | Teste: 75.78% 
Época 10 | Treino: 77.35% | Teste: 77.78% 
Época 11 | Treino: 75.59% | Teste: 80.91% 
Época 12 | Treino: 74.93% | Teste: 75.50% 
Época 13 | Treino: 77.81% | Teste: 79.20% 
Época 14 | Treino: 74.35% | Teste: 75.78% 
Época 15 | Treino: 77.68% | Teste: 80.63% 
Época 16 | Treino: 79.83% | Teste: 74.93% 
Época 17 | Treino: 77.22% | Teste: 76.35% 
Época 18 | Treino: 78.46% | Teste: 82.91% 🏆 NOVO RECORDE
Época 19 | Treino: 78.66% | Teste: 79.20% 
Época 20 | Treino: 78.72% | Teste: 82.62% 
Época 21 | Treino: 80.48% | Teste: 81.48% 
Época 22 | Tre

In [6]:
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize

model_mlsd.load_state_dict(torch.load('melhor_modelo_ConvKANWithResNet_MLSD.pth'))
model_mlsd.eval()

y_true, y_pred, y_prob_both = [], [], []
with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        outputs = model_mlsd(images)
        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_prob_both.extend(probabilities.cpu().numpy())

y_true, y_pred, y_prob_both = np.array(y_true), np.array(y_pred), np.array(y_prob_both)

# Binarização para MLSD (OVR)
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
classes_names = ['Encostado (0)', 'Curta (1)', 'Distância (2)']

cm = confusion_matrix(y_true, y_pred)
FP, FN, TP = cm.sum(axis=0) - np.diag(cm), cm.sum(axis=1) - np.diag(cm), np.diag(cm)
TN = cm.sum() - (FP + FN + TP)
support = cm.sum(axis=1)

acc = accuracy_score(y_true, y_pred)
metrics = {}
for avg in ['macro', 'micro', 'weighted']:
    metrics[f'Prec_{avg}'] = precision_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'Rec_{avg}'] = recall_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'F1_{avg}'] = f1_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'AUC_{avg}'] = roc_auc_score(y_true_bin, y_prob_both, multi_class='ovr', average=avg)

spec = np.divide(TN, (TN + FP), out=np.zeros_like(TN, dtype=float), where=(TN + FP)!=0)
metrics['Spec_macro'], metrics['Spec_weighted'] = np.mean(spec), np.average(spec, weights=support)

print("\n TABELA 5 - PERFORMANCE METRICS (MLSD)")
print("="*120)
print(f"{'ConvKAN+ResNet':<14} {'MLSD':<10} {acc:<7.3f} | F1 Macro: {metrics['F1_macro']:.3f} | AUC Macro: {metrics['AUC_macro']:.3f}")
print("="*120 + "\n")
print(classification_report(y_true, y_pred, target_names=classes_names, zero_division=0))


 TABELA 5 - PERFORMANCE METRICS (MLSD)
ConvKAN+ResNet MLSD       0.829   | F1 Macro: 0.465 | AUC Macro: 0.793

               precision    recall  f1-score   support

Encostado (0)       0.00      0.00      0.00         9
    Curta (1)       0.62      0.40      0.48        60
Distância (2)       0.88      0.95      0.91       282

     accuracy                           0.83       351
    macro avg       0.50      0.45      0.47       351
 weighted avg       0.81      0.83      0.82       351

